# Optimized leakage-safe dengue differential-diagnosis analysis

This notebook implements the complete second-round reviewer analysis. It uses patient-disjoint splitting, grouped cross-validation, fold-specific imputation/scaling/Borderline-SMOTE, patient bootstrap intervals, first-measurement and sensitivity analyses, paired model comparison, and optional full-stack XAI.

This optimized version uses randomized hyperparameter search and prints elapsed time for every outer fold and model. **Run the cells in order.**

## 1. Install packages

Run this cell once whenever a fresh Google Colab session starts.

In [1]:
%pip install -q pandas numpy scipy scikit-learn imbalanced-learn==0.14.2 xgboost matplotlib seaborn joblib shap lime psutil

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Configure the run

The defaults below perform the final analysis because the quick verification run has already succeeded.

In [2]:
CSV_PATH = '/content/figsharecsv.csv'
OUTPUT_DIR = '/content/second_review_results_optimized'
BOOTSTRAP = 2000
QUICK = False
RUN_XAI = True

print('Dataset:', CSV_PATH)
print('Output:', OUTPUT_DIR)
print('Quick verification:', QUICK)
print('Bootstrap samples:', BOOTSTRAP)
print('Run XAI:', RUN_XAI)

Dataset: /content/figsharecsv.csv
Output: /content/second_review_results_optimized
Quick verification: False
Bootstrap samples: 2000
Run XAI: True


## 3. Complete analysis code

The following cells contain the complete commented implementation. Do not run an individual later cell before running all earlier definition cells.

In [3]:
"""
SECOND-ROUND, LEAKAGE-SAFE DENGUE DIFFERENTIAL-DIAGNOSIS ANALYSIS
================================================================

This file is intentionally separate from the original analysis.  It addresses
the second-round reviewer requests while preserving the previous code as an
audit record.

Main safeguards
---------------
1. Patients, rather than individual daily observations, are split 80:20.
2. Inner and outer cross-validation use StratifiedGroupKFold.
3. Median imputation, scaling and Borderline-SMOTE are inside an imbalanced-
   learn Pipeline, so they are fitted only on each training fold.
4. Stacking uses nested, group-disjoint out-of-fold base-model probabilities.
5. Confidence intervals resample patients, not individual observations.
6. The primary analysis uses the first available measurement per patient.
7. Sensitivity analyses examine longitudinal rows, high-missingness predictors,
   and confirmed dengue versus confirmed CABI only.
8. XGBoost and stacking are compared with paired patient bootstrap intervals
   and an exact McNemar test.
9. Optional SHAP and LIME explain the complete fitted stacking probability
   function, not only one base learner.

Recommended Google Colab installation cell
-------------------------------------------
    !pip install -q pandas numpy scipy scikit-learn imbalanced-learn==0.14.2 \
        xgboost matplotlib seaborn joblib shap lime psutil

Full paper run
--------------
    python dengue_second_review_leakage_safe.py figsharecsv.csv \
        --output-dir second_review_results --bootstrap 2000 --run-xai

Short verification run before the full run
------------------------------------------
    python dengue_second_review_leakage_safe.py figsharecsv.csv \
        --output-dir second_review_quick --bootstrap 100 --quick --skip-xai

Important: the full analysis can take considerable time because leakage-safe
stacking requires nested group-wise tuning in every out-of-fold split.
"""

from __future__ import annotations

import argparse
import importlib.metadata
import json
import math
import os
import platform
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from scipy.optimize import minimize
from scipy.stats import binomtest
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    hamming_loss,
    jaccard_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedGroupKFold,
    StratifiedShuffleSplit,
)
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier


In [4]:
# =============================================================================
# STEP 1 — REPRODUCIBILITY, COLUMN NAMES AND PREDICTORS
# =============================================================================

RANDOM_STATE = 42
# Randomized search keeps the leakage-safe nested design computationally
# practical on a standard two-CPU Google Colab runtime. The complete candidate
# ranges are still recorded, but only this many combinations are sampled in
# each inner search during the full analysis.
FULL_RANDOM_SEARCH_ITERATIONS = 8
PATIENT_COLUMN = "id"
RAW_TARGET_COLUMN = (
    "dengue or probable (suspected) CABI(CAB) or confirmed CABI(CAB)"
)
DATE_SYMPTOM = "date of first symptom"
DATE_ADMISSION = "date of admission"
DATE_SAMPLING = "date of sampling"

# Exactly the predictors declared for the revised analysis.  Absolute
# neutrophil/lymphocyte/monocyte counts and IPF count are deliberately absent.
FULL_FEATURES = [
    "age",
    "sex_male",
    "days_to_admission",
    "days_to_sampling",
    "wbc",
    "neut%",
    "lymp%",
    "mono%",
    "eosino%",
    "rbc",
    "hb",
    "plt",
    "ipf%",
    "hct",
    "crp",
    "procalcitonin",
    "tbil",
    "ast",
    "alt",
    "bun",
    "creatinin",
]

# The reviewer specifically highlighted variables with approximately 48–89%
# missingness.  The script also computes the exclusion list from the training
# data using MISSINGNESS_THRESHOLD, rather than looking at test data.
MISSINGNESS_THRESHOLD = 40.0

MODEL_ORDER = ["Random Forest", "XGBoost", "Logistic Regression"]
STACK_NAME = "Stacked ensemble"


@dataclass
class AnalysisResult:
    scenario: str
    features: list[str]
    patient_predictions: dict[str, pd.DataFrame]
    row_predictions: pd.DataFrame
    meta_model: LogisticRegression
    base_models: dict[str, Any]
    test_frame: pd.DataFrame


def parse_arguments() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("csv_path", type=Path, help="Path to figsharecsv.csv")
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("second_review_results"),
        help="Output directory for all tables, figures and fitted models",
    )
    parser.add_argument(
        "--bootstrap",
        type=int,
        default=2000,
        help="Number of patient-level bootstrap resamples (paper: 2000)",
    )
    parser.add_argument(
        "--quick",
        action="store_true",
        help="Use one parameter combination and fewer folds for a smoke test",
    )
    parser.add_argument(
        "--run-xai",
        action="store_true",
        help="Run full-stacking SHAP and LIME after the statistical analyses",
    )
    parser.add_argument(
        "--skip-xai",
        action="store_true",
        help="Explicitly skip XAI (useful during the quick verification run)",
    )
    return parser.parse_args()


def make_directories(root: Path) -> dict[str, Path]:
    paths = {
        "root": root,
        "tables": root / "tables",
        "figures": root / "figures",
        "models": root / "models",
        "xai": root / "xai",
        "logs": root / "logs",
    }
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    return paths


def save_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, indent=2, default=str), encoding="utf-8")


In [5]:
# =============================================================================
# STEP 2 — LOAD DATA AND CONSTRUCT VARIABLES WITHOUT IMPUTATION
# =============================================================================

def load_and_prepare_data(csv_path: Path) -> pd.DataFrame:
    """Load labels/dates and derive predictors; do not impute or scale here."""
    data = pd.read_csv(csv_path).copy()
    required = {
        PATIENT_COLUMN,
        RAW_TARGET_COLUMN,
        DATE_SYMPTOM,
        DATE_ADMISSION,
        DATE_SAMPLING,
        "sex",
    }
    missing_required = sorted(required.difference(data.columns))
    if missing_required:
        raise ValueError(f"Required columns are missing: {missing_required}")

    raw_label = data[RAW_TARGET_COLUMN].astype(str).str.strip().str.lower()
    allowed = {"dengue", "confirmed cab", "probable cab"}
    unexpected = sorted(set(raw_label.unique()).difference(allowed))
    if unexpected:
        raise ValueError(f"Unexpected diagnostic labels: {unexpected}")

    data["diagnostic_group"] = raw_label.map(
        {
            "dengue": "Dengue",
            "confirmed cab": "Confirmed CABI",
            "probable cab": "Suspected/probable CABI",
        }
    )
    data["is_dengue"] = raw_label.eq("dengue").astype(int)

    # Validate that one participant does not have contradictory diagnoses.
    diagnostic_counts = data.groupby(PATIENT_COLUMN)["diagnostic_group"].nunique()
    contradictory = diagnostic_counts[diagnostic_counts > 1]
    if not contradictory.empty:
        raise ValueError(
            f"{len(contradictory)} patients have contradictory diagnostic labels."
        )

    symptom = pd.to_datetime(data[DATE_SYMPTOM], dayfirst=True, errors="coerce")
    admission = pd.to_datetime(data[DATE_ADMISSION], dayfirst=True, errors="coerce")
    sampling = pd.to_datetime(data[DATE_SAMPLING], dayfirst=True, errors="coerce")
    data["symptom_date_parsed"] = symptom
    data["admission_date_parsed"] = admission
    data["sampling_date_parsed"] = sampling

    # Temporal definitions are explicit and non-overlapping.
    data["days_to_admission"] = (admission - symptom).dt.days
    data["days_to_sampling"] = (sampling - admission).dt.days
    data["illness_day_at_sampling"] = (sampling - symptom).dt.days

    sex = data["sex"].astype(str).str.strip().str.lower()
    known_sex = sex.isin(["male", "female"])
    data["sex_male"] = np.where(known_sex, sex.eq("male").astype(float), np.nan)

    for column in FULL_FEATURES:
        if column not in data.columns:
            raise ValueError(f"Modelling predictor is missing: {column}")
        data[column] = pd.to_numeric(data[column], errors="coerce")

    # Used only for reporting missingness over the clinical timeline.
    data["sampling_period"] = pd.cut(
        data["days_to_sampling"],
        bins=[-np.inf, 0, 2, np.inf],
        labels=["Admission day", "Hospital days 1–2", "Hospital day 3 or later"],
    )
    return data


def first_available_measurement(data: pd.DataFrame) -> pd.DataFrame:
    """Select the earliest available laboratory row for each patient."""
    ordered = data.copy()
    ordered["_original_order"] = np.arange(len(ordered))
    ordered = ordered.sort_values(
        [PATIENT_COLUMN, "sampling_date_parsed", "_original_order"],
        na_position="last",
    )
    first = ordered.drop_duplicates(PATIENT_COLUMN, keep="first").copy()
    return first.drop(columns="_original_order").reset_index(drop=True)


In [6]:
# =============================================================================
# STEP 3 — DESCRIPTIVE MISSINGNESS ANALYSES REQUESTED BY THE REVIEWER
# =============================================================================

def missingness_table(frame: pd.DataFrame, by: list[str] | None = None) -> pd.DataFrame:
    group_columns = by or []
    rows: list[dict[str, Any]] = []
    if group_columns:
        iterator = frame.groupby(group_columns, observed=True, dropna=False)
    else:
        iterator = [((), frame)]

    for keys, subset in iterator:
        if not isinstance(keys, tuple):
            keys = (keys,)
        group_values = dict(zip(group_columns, keys))
        for feature in FULL_FEATURES:
            rows.append(
                {
                    **group_values,
                    "feature": feature,
                    "total_n": int(len(subset)),
                    "missing_n": int(subset[feature].isna().sum()),
                    "missing_percent": float(subset[feature].isna().mean() * 100),
                }
            )
    return pd.DataFrame(rows)


def save_descriptive_outputs(data: pd.DataFrame, paths: dict[str, Path]) -> None:
    patient_labels = data.drop_duplicates(PATIENT_COLUMN)
    summary = {
        "observations": int(len(data)),
        "patients": int(data[PATIENT_COLUMN].nunique()),
        "observations_by_group": data["diagnostic_group"].value_counts().to_dict(),
        "patients_by_group": patient_labels["diagnostic_group"].value_counts().to_dict(),
        "maximum_observations_per_patient": int(data.groupby(PATIENT_COLUMN).size().max()),
    }
    save_json(paths["tables"] / "dataset_summary.json", summary)
    missingness_table(data).to_csv(
        paths["tables"] / "missingness_overall.csv", index=False
    )
    missingness_table(data, ["diagnostic_group"]).to_csv(
        paths["tables"] / "missingness_by_diagnostic_group.csv", index=False
    )
    missingness_table(data, ["sampling_period"]).to_csv(
        paths["tables"] / "missingness_by_sampling_period.csv", index=False
    )
    missingness_table(data, ["diagnostic_group", "sampling_period"]).to_csv(
        paths["tables"] / "missingness_by_group_and_sampling_period.csv", index=False
    )


In [7]:
# =============================================================================
# STEP 4 — CREATE ONE PATIENT-DISJOINT HOLDOUT PARTITION
# =============================================================================

def patient_holdout_split(data: pd.DataFrame) -> tuple[set[Any], set[Any]]:
    patients = (
        data[[PATIENT_COLUMN, "is_dengue"]]
        .drop_duplicates(PATIENT_COLUMN)
        .reset_index(drop=True)
    )
    splitter = StratifiedShuffleSplit(
        n_splits=1, test_size=0.20, random_state=RANDOM_STATE
    )
    train_position, test_position = next(
        splitter.split(patients[[PATIENT_COLUMN]], patients["is_dengue"])
    )
    train_ids = set(patients.iloc[train_position][PATIENT_COLUMN])
    test_ids = set(patients.iloc[test_position][PATIENT_COLUMN])
    overlap = train_ids.intersection(test_ids)
    if overlap:
        raise RuntimeError(f"Patient leakage found in holdout split: {len(overlap)} IDs")
    return train_ids, test_ids


def save_split_summary(
    data: pd.DataFrame,
    train_ids: set[Any],
    test_ids: set[Any],
    paths: dict[str, Path],
) -> None:
    rows = []
    for partition, ids in [("Training", train_ids), ("Held-out test", test_ids)]:
        subset = data[data[PATIENT_COLUMN].isin(ids)]
        patient_subset = subset.drop_duplicates(PATIENT_COLUMN)
        rows.append(
            {
                "partition": partition,
                "patients": int(patient_subset[PATIENT_COLUMN].nunique()),
                "dengue_patients": int(patient_subset["is_dengue"].sum()),
                "non_dengue_patients": int((1 - patient_subset["is_dengue"]).sum()),
                "observations": int(len(subset)),
                "dengue_observations": int(subset["is_dengue"].sum()),
                "non_dengue_observations": int((1 - subset["is_dengue"]).sum()),
            }
        )
    pd.DataFrame(rows).to_csv(paths["tables"] / "patient_disjoint_split.csv", index=False)
    pd.DataFrame({PATIENT_COLUMN: sorted(train_ids)}).to_csv(
        paths["tables"] / "training_patient_ids.csv", index=False
    )
    pd.DataFrame({PATIENT_COLUMN: sorted(test_ids)}).to_csv(
        paths["tables"] / "test_patient_ids.csv", index=False
    )


In [8]:
# =============================================================================
# STEP 5 — PIPELINE AND HYPERPARAMETER SEARCH SPACES
# =============================================================================

def build_pipeline(estimator: Any) -> ImbPipeline:
    """Every fitted fold learns imputation, scaling and SMOTE independently."""
    return ImbPipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=False)),
            ("scaler", StandardScaler()),
            (
                "smote",
                BorderlineSMOTE(
                    sampling_strategy="auto",
                    random_state=RANDOM_STATE,
                    k_neighbors=5,
                    m_neighbors=10,
                    kind="borderline-1",
                ),
            ),
            ("model", estimator),
        ]
    )


def model_search_spaces(quick: bool) -> dict[str, tuple[Any, dict[str, list[Any]]]]:
    spaces: dict[str, tuple[Any, dict[str, list[Any]]]] = {
        "Random Forest": (
            RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
            {
                "model__n_estimators": [100, 200],
                "model__max_depth": [20, 50, None],
                "model__min_samples_split": [2, 10],
                "model__min_samples_leaf": [1, 4],
                "model__max_features": ["sqrt", 0.5],
                "model__bootstrap": [True],
            },
        ),
        "XGBoost": (
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                tree_method="hist",
            ),
            {
                "model__n_estimators": [100, 200],
                "model__max_depth": [3, 5],
                "model__learning_rate": [0.05, 0.1],
                "model__subsample": [0.8, 1.0],
                "model__colsample_bytree": [0.8, 1.0],
                "model__min_child_weight": [1],
            },
        ),
        "Logistic Regression": (
            LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
            {
                "model__C": [0.01, 0.1, 1.0, 10.0, 100.0],
                "model__penalty": ["l2"],
                "model__solver": ["liblinear"],
            },
        ),
    }
    if quick:
        spaces["Random Forest"] = (
            spaces["Random Forest"][0],
            {
                "model__n_estimators": [100],
                "model__max_depth": [50],
                "model__min_samples_split": [2],
                "model__min_samples_leaf": [1],
                "model__max_features": ["sqrt"],
                "model__bootstrap": [True],
            },
        )
        spaces["XGBoost"] = (
            spaces["XGBoost"][0],
            {
                "model__n_estimators": [100],
                "model__max_depth": [3],
                "model__learning_rate": [0.1],
                "model__subsample": [1.0],
                "model__colsample_bytree": [1.0],
                "model__min_child_weight": [1],
            },
        )
        spaces["Logistic Regression"] = (
            spaces["Logistic Regression"][0],
            {
                "model__C": [1.0],
                "model__penalty": ["l2"],
                "model__solver": ["liblinear"],
            },
        )
    return spaces


def grouped_cv(n_splits: int, seed_offset: int = 0) -> StratifiedGroupKFold:
    return StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE + seed_offset,
    )


def fit_grid_search(
    estimator: Any,
    param_grid: dict[str, list[Any]],
    X: pd.DataFrame,
    y: pd.Series,
    groups: pd.Series,
    n_splits: int,
    seed_offset: int,
) -> RandomizedSearchCV:
    search = RandomizedSearchCV(
        estimator=build_pipeline(clone(estimator)),
        param_distributions=param_grid,
        n_iter=min(
            FULL_RANDOM_SEARCH_ITERATIONS,
            math.prod(len(values) for values in param_grid.values()),
        ),
        scoring="average_precision",
        cv=grouped_cv(n_splits, seed_offset),
        refit=True,
        n_jobs=-1,
        return_train_score=False,
        error_score="raise",
        random_state=RANDOM_STATE + seed_offset,
        verbose=1,
    )
    search.fit(X, y, groups=groups)
    return search


In [9]:
# =============================================================================
# STEP 6 — NESTED, GROUP-DISJOINT STACKING
# =============================================================================

def nested_oof_base_predictions(
    X: pd.DataFrame,
    y: pd.Series,
    groups: pd.Series,
    quick: bool,
    scenario: str,
    tables_dir: Path,
) -> tuple[np.ndarray, list[dict[str, Any]]]:
    """
    Produce meta-learner training data.

    Each outer validation fold is completely excluded from imputation, scaling,
    SMOTE, hyperparameter selection and base-model fitting.  Hyperparameters are
    selected inside the corresponding outer-training fold.
    """
    outer_splits = 3 if quick else 5
    # Three grouped inner folds provide leakage-safe hyperparameter selection
    # while keeping the full nested analysis feasible in Colab.
    inner_splits = 3
    outer = grouped_cv(outer_splits, seed_offset=100)
    spaces = model_search_spaces(quick)
    oof = np.full((len(X), len(MODEL_ORDER)), np.nan, dtype=float)
    tuning_rows: list[dict[str, Any]] = []

    for fold, (outer_train, outer_valid) in enumerate(
        outer.split(X, y, groups), start=1
    ):
        fold_start = time.time()
        print(
            f"[{scenario}] outer fold {fold}/{outer_splits} started "
            f"({time.strftime('%H:%M:%S')})",
            flush=True,
        )
        fold_train_groups = set(groups.iloc[outer_train])
        fold_valid_groups = set(groups.iloc[outer_valid])
        if fold_train_groups.intersection(fold_valid_groups):
            raise RuntimeError("Patient overlap occurred in an outer stacking fold.")

        for model_index, name in enumerate(MODEL_ORDER):
            model_start = time.time()
            print(f"  Tuning {name}...", flush=True)
            estimator, grid = spaces[name]
            search = fit_grid_search(
                estimator,
                grid,
                X.iloc[outer_train],
                y.iloc[outer_train],
                groups.iloc[outer_train],
                n_splits=inner_splits,
                seed_offset=200 + fold,
            )
            oof[outer_valid, model_index] = search.predict_proba(
                X.iloc[outer_valid]
            )[:, 1]
            tuning_rows.append(
                {
                    "scenario": scenario,
                    "outer_fold": fold,
                    "model": name,
                    "inner_cv_best_average_precision": float(search.best_score_),
                    "selected_parameters": json.dumps(search.best_params_, default=str),
                    "outer_training_patients": len(fold_train_groups),
                    "outer_validation_patients": len(fold_valid_groups),
                }
            )
            print(
                f"  {name} completed in {(time.time() - model_start) / 60:.1f} min; "
                f"fold elapsed {(time.time() - fold_start) / 60:.1f} min",
                flush=True,
            )

    if np.isnan(oof).any():
        raise RuntimeError("One or more stacking OOF probabilities were not generated.")
    pd.DataFrame(tuning_rows).to_csv(
        tables_dir / f"{scenario}_nested_oof_tuning.csv", index=False
    )
    return oof, tuning_rows


def fit_final_base_models(
    X: pd.DataFrame,
    y: pd.Series,
    groups: pd.Series,
    quick: bool,
    scenario: str,
    tables_dir: Path,
) -> dict[str, Any]:
    inner_splits = 3
    fitted: dict[str, Any] = {}
    rows = []
    for index, name in enumerate(MODEL_ORDER):
        model_start = time.time()
        print(f"Final grouped tuning for {scenario}: {name}", flush=True)
        estimator, grid = model_search_spaces(quick)[name]
        search = fit_grid_search(
            estimator,
            grid,
            X,
            y,
            groups,
            n_splits=inner_splits,
            seed_offset=500 + index,
        )
        fitted[name] = search.best_estimator_
        rows.append(
            {
                "scenario": scenario,
                "model": name,
                "selection_metric": "average_precision",
                "best_grouped_cv_average_precision": float(search.best_score_),
                "selected_parameters": json.dumps(search.best_params_, default=str),
                "complete_search_space": json.dumps(grid, default=str),
            }
        )
        print(
            f"Final {name} completed in {(time.time() - model_start) / 60:.1f} min",
            flush=True,
        )
    pd.DataFrame(rows).to_csv(
        tables_dir / f"{scenario}_final_hyperparameters.csv", index=False
    )
    return fitted


def tune_and_fit_meta_model(
    oof_probabilities: np.ndarray,
    y: pd.Series,
    groups: pd.Series,
    quick: bool,
) -> LogisticRegression:
    c_values = [1.0] if quick else [0.01, 0.1, 1.0, 10.0, 100.0]
    meta_search = GridSearchCV(
        LogisticRegression(
            penalty="l2", solver="liblinear", max_iter=5000,
            random_state=RANDOM_STATE,
        ),
        param_grid={"C": c_values},
        scoring="average_precision",
        cv=grouped_cv(3 if quick else 5, seed_offset=700),
        refit=True,
        n_jobs=-1,
    )
    meta_search.fit(oof_probabilities, y, groups=groups)
    return meta_search.best_estimator_


In [10]:
# =============================================================================
# STEP 7 — METRICS, CALIBRATION AND PATIENT-CLUSTER BOOTSTRAP
# =============================================================================

def calibration_intercept_slope(y_true: np.ndarray, probability: np.ndarray) -> tuple[float, float]:
    probability = np.clip(np.asarray(probability, dtype=float), 1e-6, 1 - 1e-6)
    y_true = np.asarray(y_true, dtype=float)
    logit = np.log(probability / (1 - probability))

    def objective(parameters: np.ndarray) -> float:
        intercept, slope = parameters
        linear = intercept + slope * logit
        fitted_probability = 1 / (1 + np.exp(-np.clip(linear, -40, 40)))
        return -np.sum(
            y_true * np.log(fitted_probability + 1e-12)
            + (1 - y_true) * np.log(1 - fitted_probability + 1e-12)
        )

    result = minimize(objective, x0=np.array([0.0, 1.0]), method="BFGS")
    if not result.success:
        warnings.warn(f"Calibration optimization warning: {result.message}")
    return float(result.x[0]), float(result.x[1])


def calculate_metrics(
    y_true: np.ndarray | pd.Series,
    probability: np.ndarray | pd.Series,
    threshold: float = 0.50,
    include_calibration: bool = True,
) -> dict[str, float]:
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(probability, dtype=float)
    prediction = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    if include_calibration:
        intercept, slope = calibration_intercept_slope(y, p)
    else:
        intercept, slope = np.nan, np.nan
    return {
        "n": int(len(y)),
        "accuracy": float(accuracy_score(y, prediction)),
        "balanced_accuracy": float(balanced_accuracy_score(y, prediction)),
        "precision_ppv": float(precision_score(y, prediction, zero_division=0)),
        "negative_predictive_value": float(tn / (tn + fn)) if tn + fn else np.nan,
        "sensitivity": float(recall_score(y, prediction, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if tn + fp else np.nan,
        "f1_score": float(f1_score(y, prediction, zero_division=0)),
        "jaccard_score": float(jaccard_score(y, prediction, zero_division=0)),
        "hamming_loss": float(hamming_loss(y, prediction)),
        "auroc": float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else np.nan,
        "auprc": float(average_precision_score(y, p)) if len(np.unique(y)) == 2 else np.nan,
        "brier_score": float(brier_score_loss(y, p)),
        "log_loss": float(log_loss(y, p, labels=[0, 1])),
        "calibration_intercept": intercept,
        "calibration_slope": slope,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def aggregate_to_patient(
    test_frame: pd.DataFrame,
    probability: np.ndarray,
) -> pd.DataFrame:
    rows = pd.DataFrame(
        {
            PATIENT_COLUMN: test_frame[PATIENT_COLUMN].to_numpy(),
            "diagnostic_group": test_frame["diagnostic_group"].to_numpy(),
            "is_dengue": test_frame["is_dengue"].to_numpy(),
            "probability": probability,
        }
    )
    patient = (
        rows.groupby(PATIENT_COLUMN, as_index=False)
        .agg(
            diagnostic_group=("diagnostic_group", "first"),
            is_dengue=("is_dengue", "first"),
            probability=("probability", "mean"),
            observation_count=("probability", "size"),
        )
    )
    patient["prediction"] = (patient["probability"] >= 0.50).astype(int)
    return patient


def patient_bootstrap_intervals(
    patient_predictions: pd.DataFrame,
    n_bootstrap: int,
    seed: int,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    metric_names = [
        "accuracy", "precision_ppv", "negative_predictive_value",
        "sensitivity", "specificity", "f1_score", "auroc", "auprc",
        "brier_score", "log_loss",
    ]
    point = calculate_metrics(
        patient_predictions["is_dengue"], patient_predictions["probability"]
    )
    bootstrap_records = []
    n_patients = len(patient_predictions)
    for _ in range(n_bootstrap):
        sample_positions = rng.integers(0, n_patients, size=n_patients)
        sample = patient_predictions.iloc[sample_positions]
        if sample["is_dengue"].nunique() < 2:
            continue
        # Calibration intercept and slope are retained as point estimates.
        # Avoid thousands of numerical optimizations during bootstrapping.
        bootstrap_records.append(
            calculate_metrics(
                sample["is_dengue"], sample["probability"],
                include_calibration=False,
            )
        )
    bootstrap = pd.DataFrame(bootstrap_records)
    return pd.DataFrame(
        [
            {
                "metric": metric,
                "point_estimate": point[metric],
                "ci_2.5_percent": bootstrap[metric].quantile(0.025),
                "ci_97.5_percent": bootstrap[metric].quantile(0.975),
                "successful_bootstrap_samples": len(bootstrap),
            }
            for metric in metric_names
        ]
    )


In [11]:
# =============================================================================
# STEP 8 — RUN ONE ANALYSIS SCENARIO
# =============================================================================

def run_scenario(
    scenario: str,
    frame: pd.DataFrame,
    features: list[str],
    train_ids: set[Any],
    test_ids: set[Any],
    quick: bool,
    n_bootstrap: int,
    paths: dict[str, Path],
) -> AnalysisResult:
    print(f"\n{'=' * 78}\nRUNNING SCENARIO: {scenario}\n{'=' * 78}")
    scenario_safe = scenario.lower().replace(" ", "_").replace("-", "_")
    train = frame[frame[PATIENT_COLUMN].isin(train_ids)].copy().reset_index(drop=True)
    test = frame[frame[PATIENT_COLUMN].isin(test_ids)].copy().reset_index(drop=True)
    if train.empty or test.empty:
        raise ValueError(f"Scenario {scenario} has an empty partition.")
    if set(train[PATIENT_COLUMN]).intersection(set(test[PATIENT_COLUMN])):
        raise RuntimeError(f"Scenario {scenario} contains patient overlap.")

    X_train, y_train = train[features], train["is_dengue"]
    X_test, y_test = test[features], test["is_dengue"]
    groups_train = train[PATIENT_COLUMN]

    oof, _ = nested_oof_base_predictions(
        X_train, y_train, groups_train, quick, scenario_safe, paths["tables"]
    )
    final_base_models = fit_final_base_models(
        X_train, y_train, groups_train, quick, scenario_safe, paths["tables"]
    )
    meta_model = tune_and_fit_meta_model(oof, y_train, groups_train, quick)

    base_test_probability = {
        name: final_base_models[name].predict_proba(X_test)[:, 1]
        for name in MODEL_ORDER
    }
    test_meta_features = np.column_stack(
        [base_test_probability[name] for name in MODEL_ORDER]
    )
    stack_probability = meta_model.predict_proba(test_meta_features)[:, 1]
    all_probability = {**base_test_probability, STACK_NAME: stack_probability}

    patient_predictions: dict[str, pd.DataFrame] = {}
    performance_rows = []
    row_output = test[
        [PATIENT_COLUMN, "diagnostic_group", "is_dengue", "days_to_sampling"]
    ].copy()

    for model_index, (name, probability) in enumerate(all_probability.items()):
        row_output[f"{name}_probability"] = probability
        row_output[f"{name}_prediction"] = (probability >= 0.50).astype(int)
        patient = aggregate_to_patient(test, probability)
        patient_predictions[name] = patient
        metrics = calculate_metrics(patient["is_dengue"], patient["probability"])
        performance_rows.append(
            {"scenario": scenario, "model": name, "evaluation_level": "patient", **metrics}
        )
        ci = patient_bootstrap_intervals(
            patient, n_bootstrap, seed=RANDOM_STATE + model_index
        )
        ci.insert(0, "model", name)
        ci.insert(0, "scenario", scenario)
        ci.to_csv(
            paths["tables"] / f"{scenario_safe}_{name.lower().replace(' ', '_')}_patient_ci.csv",
            index=False,
        )
        patient.to_csv(
            paths["tables"] / f"{scenario_safe}_{name.lower().replace(' ', '_')}_patient_predictions.csv",
            index=False,
        )

    pd.DataFrame(performance_rows).to_csv(
        paths["tables"] / f"{scenario_safe}_patient_performance.csv", index=False
    )
    row_output.to_csv(
        paths["tables"] / f"{scenario_safe}_test_observation_predictions.csv", index=False
    )

    # Explicit meta-classifier representation: three base-model probabilities only.
    coefficients = pd.DataFrame(
        {
            "meta_feature": [f"{name} dengue probability" for name in MODEL_ORDER],
            "coefficient": meta_model.coef_[0],
        }
    )
    coefficients.loc[len(coefficients)] = ["Intercept", meta_model.intercept_[0]]
    coefficients.to_csv(
        paths["tables"] / f"{scenario_safe}_meta_classifier_coefficients.csv",
        index=False,
    )

    # Confirm that the equation numerically reproduces the fitted meta-classifier.
    manual = 1 / (
        1
        + np.exp(
            -(
                float(meta_model.intercept_[0])
                + test_meta_features @ meta_model.coef_[0]
            )
        )
    )
    save_json(
        paths["tables"] / f"{scenario_safe}_analytical_equivalence.json",
        {
            "maximum_absolute_probability_difference": float(
                np.max(np.abs(manual - stack_probability))
            ),
            "note": (
                "This confirms numerical reconstruction only; it is not an "
                "independent model or external validation."
            ),
        },
    )

    joblib.dump(
        final_base_models, paths["models"] / f"{scenario_safe}_base_models.joblib"
    )
    joblib.dump(
        meta_model, paths["models"] / f"{scenario_safe}_meta_model.joblib"
    )
    return AnalysisResult(
        scenario=scenario,
        features=features,
        patient_predictions=patient_predictions,
        row_predictions=row_output,
        meta_model=meta_model,
        base_models=final_base_models,
        test_frame=test,
    )


In [12]:
# =============================================================================
# STEP 9 — PAIRED XGBOOST-versus-STACKING COMPARISON
# =============================================================================

def paired_bootstrap_model_difference(
    xgb: pd.DataFrame,
    stacking: pd.DataFrame,
    n_bootstrap: int,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    paired = xgb.merge(
        stacking,
        on=[PATIENT_COLUMN, "diagnostic_group", "is_dengue"],
        suffixes=("_xgb", "_stack"),
        validate="one_to_one",
    )
    metric_names = [
        "accuracy", "sensitivity", "specificity", "f1_score", "auroc", "auprc"
    ]
    point_xgb = calculate_metrics(paired["is_dengue"], paired["probability_xgb"])
    point_stack = calculate_metrics(paired["is_dengue"], paired["probability_stack"])

    rng = np.random.default_rng(RANDOM_STATE + 900)
    records = []
    for _ in range(n_bootstrap):
        sampled = paired.iloc[rng.integers(0, len(paired), size=len(paired))]
        if sampled["is_dengue"].nunique() < 2:
            continue
        xgb_metric = calculate_metrics(
            sampled["is_dengue"], sampled["probability_xgb"],
            include_calibration=False,
        )
        stack_metric = calculate_metrics(
            sampled["is_dengue"], sampled["probability_stack"],
            include_calibration=False,
        )
        records.append(
            {metric: stack_metric[metric] - xgb_metric[metric] for metric in metric_names}
        )
    bootstrap = pd.DataFrame(records)
    difference = pd.DataFrame(
        [
            {
                "metric": metric,
                "xgboost": point_xgb[metric],
                "stacking": point_stack[metric],
                "stacking_minus_xgboost": point_stack[metric] - point_xgb[metric],
                "difference_ci_2.5_percent": bootstrap[metric].quantile(0.025),
                "difference_ci_97.5_percent": bootstrap[metric].quantile(0.975),
            }
            for metric in metric_names
        ]
    )

    y = paired["is_dengue"].to_numpy()
    correct_xgb = (paired["probability_xgb"].to_numpy() >= 0.5).astype(int) == y
    correct_stack = (paired["probability_stack"].to_numpy() >= 0.5).astype(int) == y
    xgb_only_correct = int(np.sum(correct_xgb & ~correct_stack))
    stack_only_correct = int(np.sum(~correct_xgb & correct_stack))
    discordant = xgb_only_correct + stack_only_correct
    p_value = (
        float(binomtest(min(xgb_only_correct, stack_only_correct), discordant, 0.5).pvalue)
        if discordant
        else 1.0
    )
    mcnemar = {
        "xgboost_correct_stacking_wrong": xgb_only_correct,
        "xgboost_wrong_stacking_correct": stack_only_correct,
        "discordant_pairs": discordant,
        "exact_two_sided_p_value": p_value,
        "interpretation": (
            "A small p-value indicates a difference in paired error rates; it does "
            "not by itself establish clinical importance."
        ),
    }
    return difference, mcnemar


In [13]:
# =============================================================================
# STEP 10 — SUBGROUP PERFORMANCE FOR CONFIRMED AND SUSPECTED CABI
# =============================================================================

def comparator_subgroup_table(patient_predictions: pd.DataFrame) -> pd.DataFrame:
    """Compare dengue separately with each negative diagnostic subgroup."""
    rows = []
    for group in ["Confirmed CABI", "Suspected/probable CABI"]:
        negative_subset = patient_predictions[
            patient_predictions["diagnostic_group"] == group
        ]
        binary_subset = patient_predictions[
            patient_predictions["diagnostic_group"].isin(["Dengue", group])
        ]
        predicted_dengue = (negative_subset["probability"] >= 0.50).astype(int)
        binary_metrics = calculate_metrics(
            binary_subset["is_dengue"], binary_subset["probability"]
        )
        rows.append(
            {
                "comparator_group": group,
                "dengue_patients": int((binary_subset["is_dengue"] == 1).sum()),
                "comparator_patients": int(len(negative_subset)),
                "correctly_classified_non_dengue": int((predicted_dengue == 0).sum()),
                "incorrectly_classified_as_dengue": int((predicted_dengue == 1).sum()),
                "mean_comparator_dengue_probability": float(
                    negative_subset["probability"].mean()
                ) if len(negative_subset) else np.nan,
                **binary_metrics,
            }
        )
    return pd.DataFrame(rows)


In [14]:
# =============================================================================
# STEP 11 — FIGURES GENERATED FROM THE CORRECTED PRIMARY TEST SET
# =============================================================================

def save_primary_figures(result: AnalysisResult, paths: dict[str, Path]) -> None:
    sns.set_theme(style="whitegrid")
    predictions = result.patient_predictions

    plt.figure(figsize=(8, 7))
    for name, frame in predictions.items():
        fpr, tpr, _ = roc_curve(frame["is_dengue"], frame["probability"])
        auc = roc_auc_score(frame["is_dengue"], frame["probability"])
        plt.plot(fpr, tpr, label=f"{name} (AUROC={auc:.3f})")
    plt.plot([0, 1], [0, 1], "k--", linewidth=1)
    plt.xlabel("False-positive rate")
    plt.ylabel("True-positive rate")
    plt.title("Patient-level ROC curves: first available measurement")
    plt.legend()
    plt.tight_layout()
    plt.savefig(paths["figures"] / "primary_patient_roc.png", dpi=600)
    plt.close()

    plt.figure(figsize=(8, 7))
    for name, frame in predictions.items():
        precision, recall, _ = precision_recall_curve(
            frame["is_dengue"], frame["probability"]
        )
        ap = average_precision_score(frame["is_dengue"], frame["probability"])
        plt.plot(recall, precision, label=f"{name} (AUPRC={ap:.3f})")
    plt.xlabel("Recall (sensitivity)")
    plt.ylabel("Precision")
    plt.title("Patient-level precision–recall curves: first measurement")
    plt.legend()
    plt.tight_layout()
    plt.savefig(paths["figures"] / "primary_patient_precision_recall.png", dpi=600)
    plt.close()

    stack = predictions[STACK_NAME]
    cm = confusion_matrix(
        stack["is_dengue"], (stack["probability"] >= 0.50).astype(int), labels=[0, 1]
    )
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Non-dengue", "Dengue"],
        yticklabels=["Non-dengue", "Dengue"],
    )
    plt.xlabel("Predicted class")
    plt.ylabel("Observed class")
    plt.title("Stacked ensemble: patient-level confusion matrix")
    plt.tight_layout()
    plt.savefig(paths["figures"] / "primary_stacking_confusion_matrix.png", dpi=600)
    plt.close()

    observed, predicted = calibration_curve(
        stack["is_dengue"], stack["probability"], n_bins=8, strategy="quantile"
    )
    plt.figure(figsize=(6, 6))
    plt.plot(predicted, observed, "o-", label="Stacked ensemble")
    plt.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed dengue proportion")
    plt.title("Patient-level calibration: first measurement")
    plt.legend()
    plt.tight_layout()
    plt.savefig(paths["figures"] / "primary_stacking_calibration.png", dpi=600)
    plt.close()


In [15]:
# =============================================================================
# STEP 12 — OPTIONAL FULL-STACK SHAP AND LIME
# =============================================================================

def run_full_stack_xai(
    result: AnalysisResult,
    training_frame: pd.DataFrame,
    paths: dict[str, Path],
) -> None:
    try:
        import shap
        from lime.lime_tabular import LimeTabularExplainer
    except ImportError as exc:
        warnings.warn(f"SHAP/LIME skipped because a package is unavailable: {exc}")
        return

    features = result.features
    base_models = result.base_models
    meta = result.meta_model

    def stack_probability(raw_matrix: np.ndarray) -> np.ndarray:
        frame = pd.DataFrame(raw_matrix, columns=features)
        base_probability = np.column_stack(
            [base_models[name].predict_proba(frame)[:, 1] for name in MODEL_ORDER]
        )
        return meta.predict_proba(base_probability)[:, 1]

    # Explain a bounded sample because permutation SHAP for a complete ensemble
    # is computationally expensive.
    background_frame = training_frame[features].sample(
        min(50, len(training_frame)), random_state=RANDOM_STATE
    )
    background_values = SimpleImputer(strategy="median").fit_transform(background_frame)
    test_features = result.test_frame[features]
    imputer = SimpleImputer(strategy="median").fit(training_frame[features])
    explained_values = imputer.transform(test_features.sample(
        min(40, len(test_features)), random_state=RANDOM_STATE
    ))

    explainer = shap.Explainer(
        stack_probability,
        shap.maskers.Independent(background_values),
        algorithm="permutation",
        feature_names=features,
    )
    shap_values = explainer(explained_values, max_evals=2 * len(features) + 1)
    shap.plots.beeswarm(shap_values, max_display=15, show=False)
    plt.tight_layout()
    plt.savefig(paths["xai"] / "full_stacking_shap_beeswarm.png", dpi=600, bbox_inches="tight")
    plt.close()
    pd.DataFrame(
        {
            "feature": features,
            "mean_absolute_shap": np.abs(shap_values.values).mean(axis=0),
        }
    ).sort_values("mean_absolute_shap", ascending=False).to_csv(
        paths["xai"] / "full_stacking_shap_importance.csv", index=False
    )

    training_numeric = imputer.transform(training_frame[features])
    test_numeric = imputer.transform(result.test_frame[features])

    def lime_predict(raw_matrix: np.ndarray) -> np.ndarray:
        positive = stack_probability(raw_matrix)
        return np.column_stack([1 - positive, positive])

    lime = LimeTabularExplainer(
        training_numeric,
        feature_names=features,
        class_names=["Non-dengue", "Dengue"],
        mode="classification",
        random_state=RANDOM_STATE,
        discretize_continuous=True,
    )
    stack_patient = result.patient_predictions[STACK_NAME]
    # Explain one true positive and one true negative patient when available.
    desired = [
        ("true_positive", 1, 1),
        ("true_negative", 0, 0),
        ("false_positive", 0, 1),
        ("false_negative", 1, 0),
    ]
    lime_rows = []
    test_id_to_position = {
        patient_id: position
        for position, patient_id in enumerate(result.test_frame[PATIENT_COLUMN])
    }
    for case_name, truth, prediction in desired:
        candidates = stack_patient[
            (stack_patient["is_dengue"] == truth)
            & (stack_patient["prediction"] == prediction)
        ]
        if candidates.empty:
            continue
        patient_id = candidates.iloc[0][PATIENT_COLUMN]
        position = test_id_to_position[patient_id]
        explanation = lime.explain_instance(
            test_numeric[position], lime_predict, num_features=10, num_samples=5000
        )
        explanation.save_to_file(str(paths["xai"] / f"full_stacking_lime_{case_name}.html"))
        for rank, (rule, weight) in enumerate(explanation.as_list(), start=1):
            lime_rows.append(
                {
                    "case": case_name,
                    PATIENT_COLUMN: patient_id,
                    "rank": rank,
                    "feature_rule": rule,
                    "weight": weight,
                }
            )
    pd.DataFrame(lime_rows).to_csv(
        paths["xai"] / "full_stacking_lime_explanations.csv", index=False
    )


In [16]:
# =============================================================================
# STEP 13 — HARDWARE, SOFTWARE AND COMPLETE EXPERIMENTAL RECORD
# =============================================================================

def collect_environment() -> dict[str, Any]:
    packages = [
        "pandas", "numpy", "scipy", "scikit-learn", "imbalanced-learn",
        "xgboost", "matplotlib", "seaborn", "joblib", "shap", "lime", "psutil",
    ]
    versions = {}
    for package in packages:
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = "not installed"
    memory_gb = None
    try:
        import psutil
        memory_gb = round(psutil.virtual_memory().total / (1024 ** 3), 2)
    except ImportError:
        pass
    return {
        "python_version": sys.version,
        "platform": platform.platform(),
        "processor": platform.processor() or "not reported by operating system",
        "logical_cpu_count": os.cpu_count(),
        "total_memory_gb": memory_gb,
        "package_versions": versions,
        "random_state": RANDOM_STATE,
        "classification_threshold": 0.50,
        "note": "Add GPU model manually if the execution environment used a GPU.",
    }


In [17]:
# =============================================================================
# STEP 14 — MAIN EXECUTION ORDER
# =============================================================================

def main() -> int:
    args = parse_arguments()
    if not args.csv_path.exists():
        raise FileNotFoundError(args.csv_path)
    if args.bootstrap < 20:
        raise ValueError("Use at least 20 bootstrap samples; use 2000 for the paper.")
    if args.quick:
        args.bootstrap = min(args.bootstrap, 100)

    start = time.time()
    paths = make_directories(args.output_dir)
    save_json(paths["logs"] / "environment.json", collect_environment())

    data = load_and_prepare_data(args.csv_path)
    save_descriptive_outputs(data, paths)
    train_ids, test_ids = patient_holdout_split(data)
    save_split_summary(data, train_ids, test_ids, paths)

    first = first_available_measurement(data)
    training_first = first[first[PATIENT_COLUMN].isin(train_ids)]
    first[[
        PATIENT_COLUMN, "diagnostic_group", "days_to_admission",
        "days_to_sampling", "illness_day_at_sampling",
    ]].to_csv(paths["tables"] / "first_measurement_timing.csv", index=False)
    first.groupby("diagnostic_group", observed=True)[
        ["days_to_admission", "days_to_sampling", "illness_day_at_sampling"]
    ].agg(["count", "median", "min", "max"]).to_csv(
        paths["tables"] / "first_measurement_timing_summary.csv"
    )

    # The reduced feature set is defined using training data only.
    training_missing = training_first[FULL_FEATURES].isna().mean() * 100
    excluded_high_missing = training_missing[
        training_missing >= MISSINGNESS_THRESHOLD
    ].index.tolist()
    reduced_features = [f for f in FULL_FEATURES if f not in excluded_high_missing]
    save_json(
        paths["tables"] / "reduced_feature_definition.json",
        {
            "threshold_percent": MISSINGNESS_THRESHOLD,
            "excluded_features": excluded_high_missing,
            "retained_features": reduced_features,
            "training_missingness_percent": training_missing.to_dict(),
        },
    )

    results: dict[str, AnalysisResult] = {}

    # PRIMARY: first available measurement, all 21 predictors.
    results["primary"] = run_scenario(
        "Primary first measurement full model",
        first,
        FULL_FEATURES,
        train_ids,
        test_ids,
        args.quick,
        args.bootstrap,
        paths,
    )

    # SENSITIVITY A: first measurement without highly missing predictors.
    results["reduced"] = run_scenario(
        "First measurement reduced model",
        first,
        reduced_features,
        train_ids,
        test_ids,
        args.quick,
        args.bootstrap,
        paths,
    )

    # SENSITIVITY B: all longitudinal measurements, but patients remain disjoint.
    results["longitudinal"] = run_scenario(
        "Longitudinal patient-disjoint full model",
        data,
        FULL_FEATURES,
        train_ids,
        test_ids,
        args.quick,
        args.bootstrap,
        paths,
    )

    # SENSITIVITY C: confirmed dengue versus confirmed CABI only.
    confirmed = first[
        first["diagnostic_group"].isin(["Dengue", "Confirmed CABI"])
    ].copy()
    results["confirmed_only"] = run_scenario(
        "Confirmed dengue versus confirmed CABI",
        confirmed,
        FULL_FEATURES,
        train_ids,
        test_ids,
        args.quick,
        args.bootstrap,
        paths,
    )

    primary = results["primary"]
    paired_difference, mcnemar = paired_bootstrap_model_difference(
        primary.patient_predictions["XGBoost"],
        primary.patient_predictions[STACK_NAME],
        args.bootstrap,
    )
    paired_difference.to_csv(
        paths["tables"] / "primary_xgboost_vs_stacking_paired_difference.csv",
        index=False,
    )
    save_json(paths["tables"] / "primary_xgboost_vs_stacking_mcnemar.json", mcnemar)

    comparator_subgroup_table(
        primary.patient_predictions[STACK_NAME]
    ).to_csv(
        paths["tables"] / "primary_stacking_comparator_subgroups.csv", index=False
    )

    # One compact table for direct manuscript comparison across all scenarios.
    consolidated_rows = []
    for result in results.values():
        for model_name, patient_table in result.patient_predictions.items():
            consolidated_rows.append(
                {
                    "scenario": result.scenario,
                    "model": model_name,
                    **calculate_metrics(
                        patient_table["is_dengue"], patient_table["probability"]
                    ),
                }
            )
    pd.DataFrame(consolidated_rows).to_csv(
        paths["tables"] / "all_scenarios_patient_performance.csv", index=False
    )

    save_primary_figures(primary, paths)

    if args.run_xai and not args.skip_xai:
        run_full_stack_xai(primary, training_first, paths)

    save_json(
        paths["logs"] / "run_summary.json",
        {
            "elapsed_minutes": round((time.time() - start) / 60, 2),
            "scenarios_completed": [result.scenario for result in results.values()],
            "primary_analysis": "First available measurement per patient",
            "holdout_unit": "Patient",
            "confidence_interval_unit": "Patient",
            "classification_threshold": 0.50,
        },
    )
    print(f"\nCompleted successfully. Results: {args.output_dir.resolve()}")
    return 0


## 4. Execute the analysis

This cell passes the configuration directly to the analysis. It deliberately does not read command-line arguments, so the Colab `-f` error cannot occur.

In [18]:
from argparse import Namespace

def parse_arguments():
    return Namespace(
        csv_path=Path(CSV_PATH),
        output_dir=Path(OUTPUT_DIR),
        bootstrap=int(BOOTSTRAP),
        quick=bool(QUICK),
        run_xai=bool(RUN_XAI),
        skip_xai=not bool(RUN_XAI),
    )

analysis_status = main()
print('Analysis status:', analysis_status)


RUNNING SCENARIO: Primary first measurement full model
[primary_first_measurement_full_model] outer fold 1/5 started (08:19:44)
  Tuning Random Forest...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
  Random Forest completed in 0.2 min; fold elapsed 0.2 min
  Tuning XGBoost...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
  XGBoost completed in 0.0 min; fold elapsed 0.2 min
  Tuning Logistic Regression...
Fitting 3 folds for each of 5 candidates, totalling 15 fits
  Logistic Regression completed in 0.0 min; fold elapsed 0.2 min
[primary_first_measurement_full_model] outer fold 2/5 started (08:19:56)
  Tuning Random Forest...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
  Random Forest completed in 0.1 min; fold elapsed 0.1 min
  Tuning XGBoost...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
  XGBoost completed in 0.0 min; fold elapsed 0.2 min
  Tuning Logistic Regression...
Fitting 3 folds for each of 5 candidates, totalling 1

## 5. Download the completed results

Run this only after the analysis completes. It creates one ZIP archive containing the tables, figures, models, XAI outputs and execution logs.

In [19]:
import shutil
from google.colab import files

archive = shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
print('Created:', archive)
files.download(archive)

Created: /content/second_review_results_optimized.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>